# Avaliação Prática 1 — CIFAR-10 · executor Colab

Este notebook **não contém o experimento**. Ele apenas clona o repositório e executa os
scripts versionados em `activities/avaliacao-pratica-1/` na GPU do Colab.

A separação é deliberada: o notebook é o ambiente de execução (descartável), os scripts são
o artefato científico (versionado, revisável, citável). Um notebook com estado oculto e células
executadas fora de ordem não é reprodutível — e a entrega exige links para os scripts, não para
células.

**Antes de começar:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# 1. Confirmar a GPU alocada. Sem isso, tudo abaixo roda em CPU e leva dias.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Clonar (ou atualizar) o repositório e instalar as dependências.
#    Idempotente: reexecutar esta célula puxa a versão mais recente em vez de falhar
#    com 'destination path already exists'.
import pathlib, subprocess

REPO = pathlib.Path('/content/machine-learning-fundamentals')
ACTIVITY = REPO / 'activities' / 'avaliacao-pratica-1'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fsd-dantas/machine-learning-fundamentals.git',
                    str(REPO)], check=True)

# Falhar aqui, alto e claro, em vez de deixar o %cd falhar em silêncio e o pip ir
# procurar o requirements.txt em /content.
assert ACTIVITY.is_dir(), (
    f'{ACTIVITY} não existe no repositório publicado. Os scripts da avaliação ainda '
    'não foram enviados ao GitHub — faça o push antes de rodar este notebook.')

%cd {ACTIVITY}
!pip install -q -r requirements.txt

import tensorflow as tf, torch
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('PyTorch   ', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# 3. Materializar e conferir os splits (10k treino / 2k val / 10k teste oficial).
#    Isso grava data/cifar10_splits.npz, que TODAS as estratégias — inclusive o ViT
#    em PyTorch — leem de volta. Os dados são idênticos entre frameworks por construção.
import common
import numpy as np

splits = common.load_splits()
print(splits.summary())
for name, y in [('train', splits.y_train), ('val', splits.y_val), ('test', splits.y_test)]:
    counts = np.bincount(y, minlength=10)
    print(f'{name:>5}: {counts}  (estratificado: {len(set(counts)) == 1})')

In [ ]:
# 4. ETAPA CORE — as 5 estratégias na semente primária (~35 min no T4).
#    Ao fim desta célula a entrega JÁ é completa: há resultado para cada estratégia,
#    melhor modelo e matriz de confusão. Tudo o que vem depois fortalece, não completa.
#    Cada script grava seu JSON assim que termina: uma desconexão custa só o run em curso.
!python run_all.py --stage core
!python report.py

In [ ]:
# 5. ETAPA ABLATIONS — as quatro questões em aberto do enunciado (~70 min), 2 sementes.
#    É aqui que está o diferencial: as estratégias acima qualquer entrega tem; ablação
#    controlada, com uma variável por vez e dispersão entre sementes, não.
#    2(a) MobileNet vs ResNet50 vs InceptionV3
#    4(a) Flatten vs GlobalMaxPooling2D vs GlobalAveragePooling2D
#    4(b) Adam vs AdamW vs SGD vs RMSprop
#    4(c) política de aumento de dados (incluindo a do notebook da aula)
!python run_all.py --stage ablations
!python report.py

In [ ]:
# 6. ETAPA SEEDS — sementes 7 e 2024 para a tabela principal (~50 min).
#    Dá desvio-padrão às linhas da comparação entre estratégias. Sem isto, o relatório
#    reporta a média de uma execução só — e o report.py declara isso honestamente,
#    omitindo o desvio em vez de imprimir um zero fabricado.
#    OPCIONAL sob prazo: pule se o tempo apertar, as ablações já carregam dispersão.
!python run_all.py --stage seeds
!python report.py

In [ ]:
# 7. ETAPA BASELINE — o setup exato do notebook da aula (VGG16 + Flatten, backbone
#    congelado, sem fase 2), para quantificar o que os desvios metodológicos compraram.
#    OPCIONAL sob prazo (~10 min).
!python run_all.py --stage baseline
!python report.py

In [ ]:
# 8. RELATÓRIO — tabelas, matriz de confusão do melhor modelo e McNemar do par top-2.
!python report.py

In [ ]:
# 9. Baixar os resultados para commitar no repositório.
#    Só os JSONs e os PNGs saem daqui: são pequenos, são a evidência, e permitem
#    regenerar o relatório inteiro sem GPU. Pesos treinados não são versionados.
!zip -q -r /content/avaliacao-pratica-1-results.zip results ../../assets/img/avaliacao-pratica-1-confusion-*.png

from google.colab import files
files.download('/content/avaliacao-pratica-1-results.zip')